## Data Pre-Processing and Modeling

In [57]:
import pandas as pd
import numpy as np
import re
from nltk.stem import SnowballStemmer
from nltk.tokenize import word_tokenize

In [58]:
df = pd.read_csv("/content/app_reviews_labeled.csv")

In [59]:
df.shape

(50000, 4)

In [60]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   content        49992 non-null  object
 1   score          50000 non-null  int64 
 2   thumbsUpCount  50000 non-null  int64 
 3   label          50000 non-null  object
dtypes: int64(2), object(2)
memory usage: 1.5+ MB


In [61]:
df.isnull().sum()

,0
content,8
score,0
thumbsUpCount,0
label,0


In [62]:
df.dropna(inplace=True)

In [63]:
df.drop('score', axis=1, inplace=True)

In [64]:
df.sample(6)

,content,thumbsUpCount,label
5136,Love v क्ष ग़ यय 7ज डी66गम5455टी,0,neutral
31526,"Use Minds, Gab, Lbry etc, they are much more r...",0,positive
41968,I really like this app but since a couple days...,647,negative
31207,"It was fake news, so I uninstalled it. Plus al...",0,negative
22659,Nfd,0,neutral
37701,"Helpful, but not very up-to-date.",0,negative


In [65]:
# Convert the sentiment into the numbers
def sentiment_into_number(sentiment):
  if sentiment == 'negative':
    return 0
  elif sentiment == 'neutral':
    return 1
  elif sentiment == 'positive':
    return 2

df['label'] = df['label'].apply(sentiment_into_number)

In [66]:
df.head()

,content,thumbsUpCount,label
0,Working with this app is so difficult. Default...,566,0
1,I would give it 0 stars if possible. No option...,189,0
2,Dear Google.. I found a very critical bug..cus...,105,0
3,Worst interface ever......can't even add a new...,403,0
4,"While opening a saved contact entry, this app ...",277,1


In [67]:
import nltk
nltk.download('punkt')
import re
import unicodedata
from nltk.stem import PorterStemmer

def lower_text(text):
  """
  This function is use to lower the playstore reviews content/text
  """
  return text.lower()


def clean_text(text):
    """
    Clean Play Store review text.

    Removes:
    - HTML tags
    - URLs
    - Emojis
    - Extra whitespace
    """

    text = str(text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove emojis and other Unicode symbols
    emoji_pattern = re.compile(
        "["
        "\U0001F300-\U0001F5FF"  # Miscellaneous Symbols and Pictographs
        "\U0001F600-\U0001F64F"  # Emoticons
        "\U0001F680-\U0001F6FF"  # Transport & Map Symbols
        "\U0001F700-\U0001F77F"  # Alchemical Symbols
        "\U0001F780-\U0001F7FF"  # Geometric Shapes Extended
        "\U0001F800-\U0001F8FF"  # Supplemental Arrows-C
        "\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
        "\U0001FA00-\U0001FA6F"  # Chess Symbols
        "\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
        "\U00002702-\U000027B0"  # Dingbats
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE,
    )

    text = emoji_pattern.sub(" ", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove leading/trailing whitespace
    text = text.strip()

    return text

def apply_stemming(text):
    """Apply Snowball stemming to a sentence."""
    stemmer = PorterStemmer()
    words = word_tokenize(text)
    stemmed_words = [stemmer.stem(word) for word in words]
    return " ".join(stemmed_words)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [68]:
import nltk
nltk.download('punkt_tab', quiet=True)


def text_preprocessing(text):
  text_lower = lower_text(text)
  cleaned_content = clean_text(text_lower)
  # stemmed_text = apply_stemming(cleaned_content)

  return cleaned_content

df['content'] = df['content'].apply(text_preprocessing)

In [69]:
from sklearn.model_selection import train_test_split

X = df["content"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [70]:
X_train.shape

(39993,)

In [71]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [72]:
from sklearn.metrics import classification_report
from sklearn.svm import LinearSVC

# Initialize model with your specific parameters
model = LinearSVC(
    C=1.1709879033298216,
    tol=0.000008137560594377145,
    loss='hinge',
    fit_intercept=True,
    class_weight='balanced',
    max_iter=4324
)

# Train
model.fit(X_train_tfidf, y_train)

# Predict
y_pred = model.predict(X_test_tfidf)

# Classification report
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.85      0.89      0.87      4436
           1       0.74      0.71      0.73      2148
           2       0.90      0.87      0.88      3415

    accuracy                           0.84      9999
   macro avg       0.83      0.82      0.83      9999
weighted avg       0.84      0.84      0.84      9999



In [73]:
sample = pd.DataFrame({
    "content": [
        "Amazing app!",
        "Too many bugs and crashes every day.",
        "It is okay, nothing special.",
        "Beautiful app",
        "Absolutely love this! Best user experience ever.",
        "The recent update completely broke the login screen.",
        "It does what it says, but the UI could be better.",
        "Highly recommended! Saves me so much time daily.",
        "Total waste of time. It freezes constantly on my phone.",
        "Just downloaded it. It works fine for now.",
        "Incredibly fast and very intuitive to navigate.",
        "Extremely disappointed. Terrible customer support.",
        "An average application, standard features like others.",
        "Perfect tool! I cannot imagine my routine without it."
    ]
})

# 4. Preprocess text into a separate column (FIXED 'df' error and preserved original text)
sample['content_clean'] = sample['content'].apply(text_preprocessing)

# 5. Transform using the existing fitted vectorizer
content_tfidf = tfidf_vectorizer.transform(sample['content_clean'])

# 6. Predict and append labels (FIXED model reference)
sample['label'] = model.predict(content_tfidf)

# 7. Display results side-by-side
print("\nPredicted Sample Sentiments:")


Predicted Sample Sentiments:


In [74]:
sample[['content', 'label']]

,content,label
0,Amazing app!,2
1,Too many bugs and crashes every day.,0
2,"It is okay, nothing special.",1
3,Beautiful app,2
4,Absolutely love this! Best user experience ever.,2
5,The recent update completely broke the login s...,0
6,"It does what it says, but the UI could be better.",2
7,Highly recommended! Saves me so much time daily.,2
8,Total waste of time. It freezes constantly on ...,0
9,Just downloaded it. It works fine for now.,2


In [23]:
# 2. Define the mapping dictionary
class_map = {0: 'negative', 1: 'neutral', 2: 'positive'}

# 3. Create DataFrame preserving the original X_test index
df_results = pd.DataFrame({
    'Text_Content': X_test,
    'Predicted_Label': y_pred
})

# 4. Map the numeric labels to text sentiments
df_results['Predicted_Label'] = df_results['Predicted_Label'].map(class_map)

In [25]:
percentages = df_results['Predicted_Label'].value_counts(normalize=True) * 100

In [26]:
percentages

,proportion
Predicted_Label,
negative,49.274160
positive,29.448362
neutral,21.277478


## Clustering

In [28]:
!pip install sentence-transformers umap-learn hdbscan keybert -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 160.1 kB/s eta 0:00:00


In [32]:
df_predicted = pd.DataFrame({
    'content': X_test,
    'label': y_pred
})

# 2. Define the mapping dictionary
class_map = {0: 'negative', 1: 'neutral', 2: 'positive'}

df_predicted['sentiment'] = df_predicted['label'].map(class_map)

In [37]:
df_predicted.sample(5)

,content,label,sentiment
1312,kindly change the contact pics as full screen ...,1,neutral
6951,it does not work on the watch without consider...,1,neutral
5448,fast when u update app,1,neutral
7592,🏻🏻🏻,1,neutral
10574,it is fab nice app i can download movies with ...,2,positive


In [50]:
# import pandas as pd
# import numpy as np
# import warnings
# warnings.filterwarnings("ignore")
# from sentence_transformers import SentenceTransformer
# import umap
# import hdbscan
# from keybert import KeyBERT
# import json
# from typing import Optional, Dict, List, Any
# import hashlib

# class SentimentTopicClusterer:
#     """
#     Clusters reviews by sentiment and extracts keywords/topics per cluster.

#     Expected DataFrame columns:
#         - content  : review text
#         - sentiment: predicted sentiment label (e.g. "negative", "neutral", "positive")
#     """

#     def __init__(
#         self,
#         embedding_model_name: str = "all-MiniLM-L6-v2",
#         top_n_keywords: int = 3,
#         min_reviews_to_cluster: int = 30,
#         top_topics_to_show: int = 5,          # for user-facing summary
#         enable_embedding_cache: bool = True,  # in-memory cache
#     ):
#         self.top_n_keywords = top_n_keywords
#         self.min_reviews_to_cluster = min_reviews_to_cluster
#         self.top_topics_to_show = top_topics_to_show
#         self.enable_embedding_cache = enable_embedding_cache

#         print("Loading embedding + keyword models...")
#         self.embedder = SentenceTransformer(embedding_model_name)
#         self.kw_model = KeyBERT(self.embedder)

#         # In-memory embedding cache: key = hash of texts → embeddings
#         self._embedding_cache: Dict[str, np.ndarray] = {}

#         self.df = None
#         self.embeddings = None
#         self.results_by_sentiment = {}
#         self.topics_by_sentiment = {}
#         self.df_final = None
#         self.app_summary = {}
#         self.user_facing_summary = {}   # clean summary for the extension

#     # ------------------------------------------------------------------
#     # Helper: create a stable cache key from the list of texts
#     # ------------------------------------------------------------------
#     def _make_cache_key(self, texts: List[str]) -> str:
#         joined = "||".join(texts)
#         return hashlib.md5(joined.encode("utf-8")).hexdigest()

#     # ------------------------------------------------------------------
#     # Main entry point
#     # ------------------------------------------------------------------
#     def fit(self, df: pd.DataFrame):
#         """
#         Run the full pipeline on the given DataFrame.
#         """
#         if not {"content", "sentiment"}.issubset(df.columns):
#             raise ValueError("DataFrame must contain 'content' and 'sentiment' columns.")

#         self.df = df.copy()
#         self.df["content"] = self.df["content"].astype(str)
#         texts = self.df["content"].tolist()

#         # ---- Embeddings with optional caching ----
#         cache_key = self._make_cache_key(texts) if self.enable_embedding_cache else None

#         if self.enable_embedding_cache and cache_key in self._embedding_cache:
#             print("Using cached embeddings...")
#             self.embeddings = self._embedding_cache[cache_key]
#         else:
#             print("Embedding all reviews...")
#             self.embeddings = self.embedder.encode(
#                 texts,
#                 batch_size=64,
#                 show_progress_bar=True,
#             )
#             if self.enable_embedding_cache and cache_key is not None:
#                 self._embedding_cache[cache_key] = self.embeddings

#         self.df["_embedding_idx"] = range(len(self.df))

#         # ---- Cluster per sentiment ----
#         self.results_by_sentiment = {}
#         self.topics_by_sentiment = {}

#         for sentiment in ["negative", "neutral", "positive"]:
#             clustered_df, topics = self._cluster_sentiment_group(sentiment)
#             self.results_by_sentiment[sentiment] = clustered_df
#             self.topics_by_sentiment[sentiment] = topics

#         # ---- Combine results ----
#         valid = [v for v in self.results_by_sentiment.values() if v is not None]
#         self.df_final = pd.concat(valid, ignore_index=True) if valid else pd.DataFrame()

#         # ---- Build keyword summary (raw) ----
#         self.app_summary = {}
#         for sentiment, topics in self.topics_by_sentiment.items():
#             keyword_list = []
#             for cluster_id, name in topics.items():
#                 if cluster_id == -1:
#                     continue
#                 keyword_list.extend(name.split(" | "))
#             self.app_summary[sentiment] = keyword_list

#         # ---- Build clean user-facing summary (top topics + counts) ----
#         self._build_user_facing_summary()

#         return self

#     # ------------------------------------------------------------------
#     # Clustering for one sentiment
#     # ------------------------------------------------------------------
#     def _cluster_sentiment_group(self, sentiment_label: str):
#         subset = self.df[self.df["sentiment"] == sentiment_label]
#         n = len(subset)

#         if n < self.min_reviews_to_cluster:
#             print(
#                 f"[{sentiment_label}] Too few reviews ({n}) to cluster meaningfully. Skipping."
#             )
#             return None, {}

#         sub_embeddings = self.embeddings[subset["_embedding_idx"].values]

#         # ---- UMAP ----
#         reducer = umap.UMAP(
#             n_components=min(10, n - 2),
#             n_neighbors=min(15, n - 1),
#             min_dist=0.0,
#             metric="cosine",
#             random_state=42,
#         )
#         reduced = reducer.fit_transform(sub_embeddings)

#         # ---- HDBSCAN ----
#         min_cluster_size = int(np.clip(n * 0.02, 15, 80))
#         clusterer = hdbscan.HDBSCAN(
#             min_cluster_size=min_cluster_size,
#             min_samples=max(5, min_cluster_size // 5),
#             metric="euclidean",
#             cluster_selection_method="eom",
#         )
#         cluster_labels = clusterer.fit_predict(reduced)

#         subset = subset.copy()
#         subset["cluster"] = cluster_labels

#         n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
#         n_noise = (cluster_labels == -1).sum()
#         print(
#             f"\n[{sentiment_label.upper()}] {n} reviews -> {n_clusters} clusters, "
#             f"{n_noise} noise ({n_noise / n * 100:.1f}%)"
#         )

#         # ---- KeyBERT keyword extraction ----
#         cluster_names = {}
#         for cluster_id in sorted(set(cluster_labels)):
#             if cluster_id == -1:
#                 cluster_names[-1] = "Uncategorized"
#                 continue

#             cluster_reviews = (
#                 subset[subset["cluster"] == cluster_id]["content"].tolist()
#             )
#             combined_text = " ".join(cluster_reviews[:200])[:20000]

#             try:
#                 keywords = self.kw_model.extract_keywords(
#                     combined_text,
#                     keyphrase_ngram_range=(1, 2),
#                     stop_words="english",
#                     top_n=self.top_n_keywords,
#                     use_mmr=True,
#                     diversity=0.5,
#                 )
#                 topic_name = (
#                     " | ".join([kw[0] for kw in keywords]) if keywords else "N/A"
#                 )
#             except Exception:
#                 topic_name = "N/A"

#             cluster_names[cluster_id] = topic_name
#             print(
#                 f"  Cluster {cluster_id:2d} ({len(cluster_reviews):4d} reviews): {topic_name}"
#             )

#         subset["topic"] = subset["cluster"].map(cluster_names)
#         return subset, cluster_names

#     # ------------------------------------------------------------------
#     # Build clean summary for the extension UI
#     # ------------------------------------------------------------------
#     def _build_user_facing_summary(self):
#         """
#         Creates a clean structure ready for the extension:

#         {
#           "negative": [
#             {"topic": "battery life | charging", "count": 42, "percentage": 18.5},
#             ...
#           ],
#           "neutral": [...],
#           "positive": [...]
#         }
#         """
#         self.user_facing_summary = {}

#         for sentiment in ["negative", "neutral", "positive"]:
#             clustered_df = self.results_by_sentiment.get(sentiment)

#             if clustered_df is None or clustered_df.empty:
#                 self.user_facing_summary[sentiment] = []
#                 continue

#             total = len(clustered_df)

#             # Count reviews per topic (ignore Uncategorized / -1)
#             topic_counts = (
#                 clustered_df[clustered_df["cluster"] != -1]
#                 .groupby("topic")
#                 .size()
#                 .reset_index(name="count")
#             )

#             # Sort by count descending and keep top N
#             topic_counts = topic_counts.sort_values("count", ascending=False)
#             topic_counts = topic_counts.head(self.top_topics_to_show)

#             topics_list = []
#             for _, row in topic_counts.iterrows():
#                 topics_list.append({
#                     "topic": row["topic"],
#                     "count": int(row["count"]),
#                     "percentage": round(row["count"] / total * 100, 1)
#                 })

#             self.user_facing_summary[sentiment] = topics_list

#     # ------------------------------------------------------------------
#     # Public helpers
#     # ------------------------------------------------------------------
#     def get_summary(self) -> dict:
#         """Raw keyword list per sentiment (legacy)."""
#         return self.app_summary

#     def get_user_facing_summary(self) -> dict:
#         """Clean summary for the extension UI (recommended)."""
#         return self.user_facing_summary

#     def print_summary(self):
#         """Pretty-print the user-facing summary."""
#         print(json.dumps(self.user_facing_summary, indent=2))

#     def get_final_dataframe(self) -> pd.DataFrame:
#         """Return the combined clustered DataFrame."""
#         return self.df_final

#     def clear_embedding_cache(self):
#         """Clear the in-memory embedding cache if needed."""
#         self._embedding_cache.clear()
#         print("Embedding cache cleared.")

In [52]:
# clusterer = SentimentTopicClusterer(
#     top_n_keywords=3,
#     top_topics_to_show=5,          # show top 5 topics
#     enable_embedding_cache=True    # cache embeddings
# )

# clusterer.fit(df_predicted)   # df must have columns: content, sentiment

# # What the extension should use:
# summary = clusterer.get_user_facing_summary()
# print(json.dumps(summary, indent=2))

Loading embedding + keyword models...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding all reviews...


Batches:   0%|          | 0/38 [00:00<?, ?it/s]


[NEGATIVE] 1188 reviews -> 13 clusters, 168 noise (14.1%)
  Cluster  0 ( 105 reviews): synced calendar | unable dark | google unreliable
  Cluster  1 ( 119 reviews): alarms work | assistant settings | google pixel
  Cluster  2 ( 121 reviews): chrome unusable | switching tabs | app search
  Cluster  3 (  72 reviews): books app | searches return | upgraded obscure
  Cluster  4 ( 143 reviews): contactless payment | cards app | frustrating google
  Cluster  5 (  84 reviews): contacts reappear | number app | freezes search
  Cluster  6 (  60 reviews): hangouts stopping | app communicate | zero star
  Cluster  7 (  74 reviews): updates app | download whatsapp | sync stops
  Cluster  8 (  46 reviews): apps stop | issue contacts | play swrvices
  Cluster  9 (  82 reviews): secure app | earth google | new samsung
  Cluster 10 (  46 reviews): calls recording | app stops | announcement annoying
  Cluster 11 (  24 reviews): google voice | chrome accent | update erratic
  Cluster 12 (  44 reviews)

In [53]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio


# Colors
colors = {
    "negative": "#EF4444",   # red
    "neutral":  "#6B7280",   # gray
    "positive": "#22C55E"    # green
}

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["Negative", "Neutral", "Positive"],
    horizontal_spacing=0.12
)

for i, sentiment in enumerate(["negative", "neutral", "positive"], start=1):
    data = summary[sentiment]

    # Shorten long topic names for readability
    topics = [t["topic"][:45] + "..." if len(t["topic"]) > 45 else t["topic"] for t in data]
    percentages = [t["percentage"] for t in data]
    counts = [t["count"] for t in data]

    # Reverse so highest is on top
    topics = topics[::-1]
    percentages = percentages[::-1]
    counts = counts[::-1]

    fig.add_trace(
        go.Bar(
            y=topics,
            x=percentages,
            orientation="h",
            marker_color=colors[sentiment],
            text=[f"{p}% ({c})" for p, c in zip(percentages, counts)],
            textposition="auto",
            name=sentiment.capitalize(),
            hovertemplate="<b>%{y}</b><br>%{x}% of reviews<br>Count: %{customdata}<extra></extra>",
            customdata=counts
        ),
        row=1, col=i
    )

    fig.update_xaxes(title_text="% of reviews", row=1, col=i, range=[0, max(percentages)*1.25])

fig.update_layout(
    title_text="Top Topics by Sentiment",
    height=480,
    width=1200,
    showlegend=False,
    margin=dict(l=20, r=20, t=60, b=40),
    font=dict(size=12)
)

fig.show()

# Optional: save as interactive HTML (good for extension)
# fig.write_html("topic_summary.html")